# EDA rápido y feature engineering — Competencia AP 2026 Q2

Notebook para descargar los datos directamente de Kaggle, explorar la variable objetivo `condition` (new/used), visualizar señales y generar una tabla con variables derivadas. No entrena un modelo ni genera una submission.

**Antes de ejecutar:** en Kaggle, aceptar las reglas de la competencia. En Google Colab, agregar el token de Kaggle como un Secret llamado `KAGGLE_API_TOKEN` y habilitar el acceso del notebook a ese secret. Se crea desde Kaggle → Settings → API → Generate New Token. No escribas el token dentro del notebook.

Ejecutar de arriba hacia abajo. Los gráficos muestran asociaciones, no causalidad. Columnas como `sold_quantity`, `status` y `last_updated` podrían reflejar información posterior al alta; se auditan, pero no se presentan como features automáticamente seguras.

## 1. Instalación, autenticación y descarga

`kagglehub` descarga y descomprime el archivo de la competencia en `/content` (almacenamiento temporal de Colab). Si el acceso falla, comprobar primero que se aceptaron las reglas y que el secret está habilitado.

In [ ]:
%pip -q install kagglehub

In [ ]:
from pathlib import Path
import os
import kagglehub

COMPETITION = "ap-2026-q-2"

# En Colab, montar Drive para conservar datos y experimentos entre sesiones.
DRIVE_ROOT = None
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    candidate_drive = Path("/content/drive/MyDrive")
    if candidate_drive.exists():
        DRIVE_ROOT = candidate_drive
except ImportError:
    pass  # Fuera de Colab, usar el almacenamiento local disponible.

if DRIVE_ROOT is not None:
    PROJECT_DIR = DRIVE_ROOT / "Analisis_Predictivo"
    DATA_DIR = PROJECT_DIR / "kaggle_cache" / "ap_2026_q2"
else:
    PROJECT_DIR = Path("/content/Analisis_Predictivo") if Path("/content").exists() else Path.cwd()
    DATA_DIR = Path("/content/ap_2026_q2") if Path("/content").exists() else PROJECT_DIR / "kaggle_cache" / "ap_2026_q2"
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Carga segura del token almacenado en Colab Secrets.
token = os.environ.get("KAGGLE_API_TOKEN")
try:
    from google.colab import userdata
    token = userdata.get("KAGGLE_API_TOKEN") or token
    if token:
        os.environ["KAGGLE_API_TOKEN"] = token
except ImportError:
    pass  # Fuera de Colab, kagglehub puede usar el token ya configurado localmente.
except Exception as exc:
    raise RuntimeError(
        "No pude leer el Secret KAGGLE_API_TOKEN. En Colab, revisa el panel Secrets, "
        "crea ese Secret y habilita el acceso para este notebook."
    ) from exc

print("Token KAGGLE_API_TOKEN disponible:", bool(token))
if not token:
    print("No encontre el token en el entorno ni en Colab Secrets. KaggleHub intentara usar otra autenticacion configurada.")

# Reutilizar los archivos persistidos si ya fueron descargados.
cached_train = next(DATA_DIR.rglob("train_data.jsonlines"), None)
cached_test = next(DATA_DIR.rglob("test_data.jsonlines"), None)
if cached_train is not None and cached_test is not None:
    downloaded_dir = DATA_DIR
    print("Reutilizo train/test ya descargados.")
else:
    try:
        downloaded_dir = Path(kagglehub.competition_download(
            COMPETITION, output_dir=str(DATA_DIR), force_download=False
        ))
    except Exception as exc:
        detail = str(exc)
        if token:
            detail = detail.replace(token, "[TOKEN OCULTO]")
        print(f"Error original de KaggleHub: {type(exc).__name__}: {detail}")
        print("Revisa: 1) aceptar las reglas de la competencia; "
              "2) que el Secret se llame exactamente KAGGLE_API_TOKEN y este habilitado; "
              "3) que el token siga vigente.")
        raise

print("Archivos disponibles en:", downloaded_dir)
for path in sorted(downloaded_dir.rglob("*")):
    if path.is_file():
        print(f"{path.relative_to(downloaded_dir)} - {path.stat().st_size / 1024**2:.1f} MiB")


## 2. Lectura y chequeos iniciales

Los datos están en JSON Lines: cada línea es una publicación. Confirmamos que los dos archivos existan y que `condition` solo esté en train.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

TARGET = "condition"
TRAIN_PATH = next(downloaded_dir.rglob("train_data.jsonlines"), None)
TEST_PATH = next(downloaded_dir.rglob("test_data.jsonlines"), None)
if TRAIN_PATH is None or TEST_PATH is None:
    raise FileNotFoundError(
        "No encontré train_data.jsonlines y test_data.jsonlines en la carpeta descargada."
    )

train = pd.read_json(TRAIN_PATH, lines=True)
test = pd.read_json(TEST_PATH, lines=True)
assert TARGET in train.columns, f"Falta el target {TARGET} en train"
assert TARGET not in test.columns, f"El target {TARGET} aparece inesperadamente en test"

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 100)
sns.set_theme(style="whitegrid", palette="Set2")

print(f"Train: {train.shape[0]:,} filas × {train.shape[1]} columnas")
print(f"Test:  {test.shape[0]:,} filas × {test.shape[1]} columnas")
print("Columnas distintas entre train y test:")
print("  solo train:", sorted(set(train.columns) - set(test.columns)))
print("  solo test: ", sorted(set(test.columns) - set(train.columns)))
display(train.head(3).T)

## 3. Target y punto de referencia

Como la métrica de competencia es Accuracy, el baseline trivial es predecir siempre la clase más frecuente. Sirve para saber cuánto debe superar cualquier modelo.

In [ ]:
target_counts = train[TARGET].value_counts(dropna=False)
target_summary = pd.DataFrame({"n": target_counts, "proporción": target_counts / len(train)})
display(target_summary)
baseline_accuracy = target_counts.max() / len(train)
print(f"Baseline de clase mayoritaria (Accuracy): {baseline_accuracy:.4f}")

fig, ax = plt.subplots(figsize=(6, 3.5))
sns.countplot(data=train, x=TARGET, order=target_counts.index, ax=ax)
ax.set(title="Distribución de condition", xlabel="condition", ylabel="Publicaciones")
for container in ax.containers:
    ax.bar_label(container)
plt.tight_layout()
plt.show()

## 4. Calidad de datos y valores faltantes

Comparamos proporciones de faltantes en train y test. Variables totalmente vacías o casi constantes suelen ser poco informativas; los faltantes en `warranty` y `parent_item_id` pueden convertirse en indicadores.

In [ ]:
def freeze_value(value):
    if isinstance(value, dict):
        return ("dict", tuple(sorted((str(k), freeze_value(v)) for k, v in value.items())))
    if isinstance(value, (list, tuple)):
        return (type(value).__name__, tuple(freeze_value(v) for v in value))
    if isinstance(value, set):
        return ("set", tuple(sorted((freeze_value(v) for v in value), key=repr)))
    if isinstance(value, np.ndarray):
        return ("array", freeze_value(value.tolist()))
    try:
        missing = pd.isna(value)
        if isinstance(missing, (bool, np.bool_)) and missing:
            return ("missing",)
    except (TypeError, ValueError):
        pass
    return ("value", type(value).__name__, repr(value))

constant_cols = [
    c for c in train.columns
    if train[c].map(freeze_value).nunique(dropna=False) <= 1
]
print("Columnas constantes o totalmente vacías en train:", constant_cols)

## 5. Categorías y relación con `condition`

Comparamos la proporción de `used` por categoría. En categorías con pocas filas, la tasa es ruidosa; por eso se muestran solo niveles frecuentes.

In [ ]:
def plot_target_by_category(df, column, top_n=15, min_rows=50):
    if column not in df:
        return
    values = df[column].astype("string").fillna("<NULO>")
    counts = values.value_counts()
    selected = counts[counts >= min_rows].head(top_n).index
    if len(selected) == 0:
        print(f"{column}: no hay grupos con al menos {min_rows} filas")
        return
    mask = values.isin(selected)
    rates = pd.crosstab(values[mask], df.loc[mask, TARGET], normalize="index")
    rates = rates.reindex(columns=[c for c in ["new", "used"] if c in rates], fill_value=0)
    rates = rates.sort_values("used" if "used" in rates else rates.columns[-1])
    ax = rates.plot(kind="barh", stacked=True, figsize=(9, max(3, .32 * len(rates))))
    ax.set(title=f"Distribución del target por {column}", xlabel="Proporción dentro del grupo", ylabel="")
    ax.legend(title=TARGET, bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    plt.show()

for col in ["category_id", "listing_type_id", "buying_mode", "currency_id",
            "accepts_mercadopago", "automatic_relist", "status"]:
    plot_target_by_category(train, col)

## 6. Precios, cantidades y distribución

Los precios tienen cola larga. Miramos `log1p(price)` para comparar rangos, y analizamos por separado la moneda más frecuente para no mezclar ARS con USD.

In [ ]:
numeric_cols = [c for c in ["price", "base_price", "original_price", "initial_quantity",
                            "available_quantity", "sold_quantity"]
               if c in train and pd.api.types.is_numeric_dtype(train[c])]
display(train[numeric_cols].describe(percentiles=[.01, .25, .5, .75, .99]).T.round(2))

if "price" in train:
    main_currency = train["currency_id"].mode().iloc[0] if "currency_id" in train else None
    price_df = train.loc[train["currency_id"].eq(main_currency) if main_currency else train.index == train.index,
                         ["price", TARGET]].dropna().copy()
    price_df["log_price"] = np.log1p(price_df["price"].clip(lower=0))
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    sns.histplot(data=price_df, x="log_price", hue=TARGET, bins=45, stat="density",
                 common_norm=False, element="step", ax=axes[0])
    sns.boxplot(data=price_df, x=TARGET, y="log_price", showfliers=False, ax=axes[1])
    axes[0].set(title=f"log(1 + precio), moneda {main_currency}", xlabel="log(1 + price)")
    axes[1].set(title="Precio por clase", ylabel="log(1 + price)")
    plt.tight_layout()
    plt.show()
    display(price_df.groupby(TARGET)["price"].describe(percentiles=[.1, .5, .9]).round(2))

if {"price", "base_price"}.issubset(train.columns):
    print("price y base_price iguales (%):", round(100 * np.isclose(train.price, train.base_price).mean(), 2))

## 7. Fechas y evolución temporal

Las fechas pueden capturar cohorte o estacionalidad. `last_updated` y `stop_time` pueden ser posteriores a la publicación; se visualizan para entenderlas, pero hay que evaluar su legitimidad según el momento de predicción.

In [ ]:
date_cols = [c for c in ["date_created", "start_time", "last_updated", "stop_time"] if c in train]
dates = train[date_cols].copy()
for col in date_cols:
    dates[col] = pd.to_datetime(dates[col], errors="coerce", utc=True)

if "date_created" in dates:
    print("Rango de date_created:", dates["date_created"].min(), "a", dates["date_created"].max())
    monthly = pd.DataFrame({"month": dates["date_created"].dt.to_period("M").astype("string"),
                            TARGET: train[TARGET]})
    monthly_counts = monthly.groupby("month").size()
    monthly_used = monthly.assign(is_used=monthly[TARGET].eq("used")).groupby("month")["is_used"].mean()
    month_plot = pd.DataFrame({"n": monthly_counts, "proporción used": monthly_used})
    month_plot = month_plot[month_plot["n"] >= 50]
    if not month_plot.empty:
        ax = month_plot["proporción used"].plot(figsize=(12, 4), marker="o")
        ax.set(title="Proporción de usados por mes de creación (n ≥ 50)",
               xlabel="Mes", ylabel="Proporción used", ylim=(0, 1))
        plt.xticks(rotation=60)
        plt.tight_layout()
        plt.show()

if {"start_time", "stop_time"}.issubset(dates.columns):
    duration_days = (dates["stop_time"] - dates["start_time"]).dt.total_seconds() / 86400
    display(duration_days.groupby(train[TARGET]).describe().round(2))

## 8. Texto de título

El título podría nombrar explícitamente el estado del producto. Medimos palabras/longitud y frases obvias en español. Si aparece una señal muy fuerte, conviene confirmar que generaliza en validación.

In [ ]:
title = train["title"].fillna("").astype(str).str.lower() if "title" in train else pd.Series("", index=train.index)
title_flags = pd.DataFrame(index=train.index)
title_flags["menciona_usado"] = title.str.contains(r"\busad[oa]s?\b|segunda mano", regex=True)
title_flags["menciona_nuevo"] = title.str.contains(r"\bnuev[oa]s?\b|sin uso|a estrenar", regex=True)
title_flags["largo_titulo"] = title.str.len()
title_flags["palabras_titulo"] = title.str.split().str.len()

for col in ["menciona_usado", "menciona_nuevo"]:
    print(f"Proporción de condition según {col}:")
    display(pd.crosstab(title_flags[col], train[TARGET], normalize="index").round(3))

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
sns.boxplot(data=title_flags.assign(condition=train[TARGET]), x=TARGET, y="largo_titulo",
            showfliers=False, ax=axes[0])
sns.boxplot(data=title_flags.assign(condition=train[TARGET]), x=TARGET, y="palabras_titulo",
            showfliers=False, ax=axes[1])
axes[0].set(title="Caracteres del título", ylabel="Caracteres")
axes[1].set(title="Palabras del título", ylabel="Cantidad")
plt.tight_layout()
plt.show()

## 9. Crear variables derivadas

Creamos features reutilizables para train y test con la misma función: logaritmos de precios, razón precio/base, componentes de fecha, longitud y frases del título, indicadores de faltantes, conteos de listas anidadas y campos de envío/ubicación.

Los campos `AUDIT_*` son diagnósticos de posible duplicación del target y no deben pasarse al modelo antes de auditarlos. El texto de `attributes` puede contener el estado de forma casi directa.

In [ ]:
import ast
import re

def parse_nested(value):
    if isinstance(value, (dict, list, tuple)):
        return value
    if isinstance(value, str):
        try:
            return ast.literal_eval(value)
        except (ValueError, SyntaxError, TypeError):
            return None
    return None

def nested_len(value):
    obj = parse_nested(value)
    if isinstance(obj, (list, tuple, dict)):
        return len(obj)
    return 0

def nested_value(value, *keys):
    obj = parse_nested(value)
    for key in keys:
        if not isinstance(obj, dict):
            return np.nan
        obj = obj.get(key)
    return obj if isinstance(obj, (str, int, float, bool)) else np.nan

def feature_engineering(df):
    out = df.copy()

    for col in ["price", "base_price"]:
        if col in out:
            x = pd.to_numeric(out[col], errors="coerce")
            out[f"{col}_log1p"] = np.log1p(x.clip(lower=0))
    if {"price", "base_price"}.issubset(out.columns):
        price = pd.to_numeric(out["price"], errors="coerce")
        base = pd.to_numeric(out["base_price"], errors="coerce").replace(0, np.nan)
        out["price_to_base_ratio"] = price / base

    for col in ["warranty", "parent_item_id", "original_price", "video_id"]:
        if col in out:
            out[f"{col}_missing"] = out[col].isna().astype("int8")

    if "title" in out:
        title = out["title"].fillna("").astype(str).str.lower()
        out["title_chars"] = title.str.len()
        out["title_words"] = title.str.split().str.len()
        out["title_mentions_used"] = title.str.contains(
            r"\busad[oa]s?\b|segunda mano", regex=True
        ).astype("int8")
        out["title_mentions_new"] = title.str.contains(
            r"\bnuev[oa]s?\b|sin uso|a estrenar", regex=True
        ).astype("int8")

    date_fields = [c for c in ["date_created", "start_time", "last_updated", "stop_time"] if c in out]
    parsed_dates = {c: pd.to_datetime(out[c], errors="coerce", utc=True) for c in date_fields}
    if "date_created" in parsed_dates:
        d = parsed_dates["date_created"]
        out["created_year"] = d.dt.year
        out["created_month"] = d.dt.month
        out["created_dayofweek"] = d.dt.dayofweek
        out["created_hour"] = d.dt.hour
    if {"start_time", "stop_time"}.issubset(parsed_dates):
        out["listing_duration_days"] = (
            parsed_dates["stop_time"] - parsed_dates["start_time"]
        ).dt.total_seconds() / 86400

    list_fields = {
        "attributes": "n_attributes", "variations": "n_variations",
        "pictures": "n_pictures", "tags": "n_tags",
        "non_mercado_pago_payment_methods": "n_other_payment_methods",
        "descriptions": "n_description_refs",
    }
    for source, feature in list_fields.items():
        if source in out:
            out[feature] = out[source].map(nested_len)

    if "shipping" in out:
        out["free_shipping"] = out["shipping"].map(lambda x: nested_value(x, "free_shipping"))
        out["local_pick_up"] = out["shipping"].map(lambda x: nested_value(x, "local_pick_up"))
        out["shipping_mode"] = out["shipping"].map(lambda x: nested_value(x, "mode"))
    if "seller_address" in out:
        out["seller_state_id"] = out["seller_address"].map(lambda x: nested_value(x, "state", "id"))
        out["seller_city_id"] = out["seller_address"].map(lambda x: nested_value(x, "city", "id"))

    def attribute_mentions_condition(value):
        obj = parse_nested(value)
        if not isinstance(obj, list):
            return np.nan
        text = " ".join(
            str(item.get(k, ""))
            for item in obj if isinstance(item, dict)
            for k in ["id", "name", "attribute_group_name", "value_name"]
        ).lower()
        terms = ["condition", "estado del artículo", "estado del inmueble", "condición"]
        return int(any(term in text for term in terms))

    if "attributes" in out:
        out["AUDIT_attribute_mentions_condition"] = out["attributes"].map(attribute_mentions_condition)
    return out

train_fe = feature_engineering(train)
test_fe = feature_engineering(test)
new_features = sorted(set(train_fe.columns) - set(train.columns))
print(f"Train con features: {train_fe.shape}; test con features: {test_fe.shape}")
print("Variables nuevas:", new_features)
display(train_fe[new_features].head())

## 10. ¿Qué señales separan las clases?

Comparamos un conjunto compacto de features nuevas contra el target. Esto ayuda a priorizar pruebas y no reemplaza una validación separada.

In [ ]:
numeric_features = [c for c in ["price_log1p", "base_price_log1p", "price_to_base_ratio",
    "title_chars", "title_words", "created_month", "created_dayofweek", "created_hour",
    "listing_duration_days", "n_attributes", "n_variations", "n_pictures", "n_tags",
    "n_other_payment_methods", "n_description_refs"] if c in train_fe]

fig, axes = plt.subplots(max(1, (len(numeric_features) + 2) // 3), 3,
                         figsize=(14, 3.5 * max(1, (len(numeric_features) + 2) // 3)))
axes = np.array(axes).reshape(-1)
for ax, col in zip(axes, numeric_features):
    sns.boxplot(data=train_fe, x=TARGET, y=col, showfliers=False, ax=ax)
    ax.set_title(col)
for ax in axes[len(numeric_features):]:
    ax.remove()
plt.tight_layout()
plt.show()

for col in ["free_shipping", "local_pick_up", "shipping_mode", "seller_state_id",
            "warranty_missing", "parent_item_id_missing", "AUDIT_attribute_mentions_condition"]:
    if col in train_fe:
        plot_target_by_category(train_fe, col, top_n=12, min_rows=50)

## 11. IDs, duplicados y posible leakage

Medimos unicidad y solapamiento train-test para `id`, `seller_id` y `parent_item_id`. Solapamiento de vendedor/familia puede ayudar a elegir la validación (aleatoria o agrupada), pero no implica automáticamente fuga.

In [ ]:
rows = []
for col in ["id", "seller_id", "parent_item_id"]:
    if col not in train:
        continue
    tr = train[col].dropna().astype(str)
    row = {"columna": col, "únicos_train": tr.nunique(),
           "duplicados_train": int(tr.duplicated().sum())}
    if col in test:
        te = test[col].dropna().astype(str)
        overlap = set(tr) & set(te)
        row["únicos_test"] = te.nunique()
        row["solapamiento_train_test"] = len(overlap)
        row["% test solapado"] = 100 * len(overlap) / max(1, te.nunique())
    rows.append(row)
display(pd.DataFrame(rows))

if "parent_item_id" in train:
    families = train.dropna(subset=["parent_item_id"]).groupby("parent_item_id")[TARGET].nunique()
    print("Familias parent_item_id con más de una clase:", int(families.gt(1).sum()), "de", len(families))

audit_fields = [c for c in ["sold_quantity", "available_quantity", "status", "last_updated",
                            "stop_time", "AUDIT_attribute_mentions_condition"] if c in train_fe]
print("Campos para auditoría temporal/duplicación del objetivo:", audit_fields)
print("Mirar su comportamiento es distinto de decidir que sean válidos: definir primero cuándo se supone que se predice condition.")

## 12. Resumen para decidir próximos experimentos

- Comparar los modelos con la misma partición estratificada y reportar Accuracy frente al baseline de clase mayoritaria.
- Probar `title` con TF-IDF; contrastarlo con un modelo tabular que use precio, categoría, listing type, ubicación, flags y conteos.
- Evaluar las variables de alta cardinalidad (`category_id`, seller/parent) con codificación dentro del fold; nunca calcular target encoding antes de separar la validación.
- Hacer ablaciones: con/sin texto, IDs, atributos que puedan copiar `condition` y campos que reflejen actividad posterior (`sold_quantity`, `status`, `last_updated`).
- Revisar que train y test tengan distribuciones comparables y que la validación imite cómo fueron separados los registros.

Al terminar, `train_fe` y `test_fe` contienen las columnas originales más las variables derivadas. Las originales permanecen para poder excluirlas o procesarlas por separado en el modelado.

## 13. Validación y comparación de modelos

Se reserva un holdout estratificado del 20% del train y se mantiene intacto durante el ajuste. La partición aleatoria es la estimación principal si el test proviene del mismo universo. Los finalistas se comparan además con 3 folds dentro del 80% de desarrollo y con un split agrupado por vendedor como estrés de generalización. La métrica oficial es Accuracy; balanced accuracy y macro-F1 muestran si una clase queda relegada.

Para reducir leakage, el primer benchmark excluye id, parent_item_id, atributos que puedan copiar condition y campos de actividad posterior como last_updated, stop_time, status y sold_quantity. El título entra mediante TF-IDF. Todas las transformaciones se ajustan dentro de cada pipeline y de cada fold.


In [ ]:
from time import perf_counter
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
                             classification_report, ConfusionMatrixDisplay)
from sklearn.model_selection import (train_test_split, StratifiedKFold,
                                     GroupShuffleSplit, cross_validate)
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.svm import LinearSVC

SEED = 42
y_raw = train[TARGET].astype(str)
label_encoder = LabelEncoder().fit(y_raw)
y = label_encoder.transform(y_raw)
classes = label_encoder.classes_
if len(classes) != 2:
    raise ValueError(f"Se esperaban dos clases, encontré {classes.tolist()}")
print("Clases:", dict(zip(classes, np.bincount(y))))
print("Baseline mayoritario:", round(max(np.bincount(y)) / len(y), 4))


### 13.1 Variables y preprocesamiento

Se usan variables numéricas derivadas, categoría, tipo de publicación, moneda, modalidad, garantía y algunas señales de envío. Los IDs se excluyen del benchmark inicial para que la métrica no dependa de memorizar publicaciones o vendedores.

El límite de términos TF-IDF controla la dimensionalidad. PCA estándar densificaría esta matriz dispersa y elevaría mucho el costo; una prueba de reducción adicional tendría que usar TruncatedSVD y validarse como otro modelo.


In [ ]:
NUMERIC_CANDIDATES = [
    "price_log1p", "base_price_log1p", "price_to_base_ratio",
    "initial_quantity", "available_quantity", "title_chars", "title_words",
    "title_mentions_used", "title_mentions_new", "created_year",
    "created_month", "created_dayofweek", "created_hour",
    "listing_duration_days", "n_attributes", "n_variations", "n_pictures",
    "n_tags", "n_other_payment_methods", "n_description_refs",
    "warranty_missing", "parent_item_id_missing", "original_price_missing",
    "video_id_missing", "free_shipping", "local_pick_up"
]
CATEGORY_CANDIDATES = [
    "category_id", "listing_type_id", "buying_mode", "currency_id",
    "accepts_mercadopago", "automatic_relist", "warranty",
    "shipping_mode", "seller_state_id"
]
numeric_features = [c for c in NUMERIC_CANDIDATES if c in train_fe]
category_features = [c for c in CATEGORY_CANDIDATES if c in train_fe]
if "title" not in train_fe:
    raise KeyError("Falta title.")

def modeling_frame(df):
    cols = ["title"] + numeric_features + category_features
    out = df.reindex(columns=list(dict.fromkeys(cols))).copy()
    out["title"] = out["title"].fillna("").astype(str)
    for c in numeric_features:
        out[c] = pd.to_numeric(out[c], errors="coerce").replace([np.inf, -np.inf], np.nan)
    for c in category_features:
        out[c] = out[c].astype("string").fillna("__MISSING__").astype(str)
    return out

X = modeling_frame(train_fe)
X_test_model = modeling_frame(test_fe)
X_dev, X_holdout, y_dev, y_holdout = train_test_split(
    X, y, test_size=0.20, random_state=SEED, stratify=y
)
print("Desarrollo:", X_dev.shape, "| holdout:", X_holdout.shape)
display(pd.DataFrame({
    "desarrollo": pd.Series(y_dev).value_counts(normalize=True).sort_index(),
    "holdout": pd.Series(y_holdout).value_counts(normalize=True).sort_index()
}, index=range(len(classes))).rename(index=dict(enumerate(classes))).round(4))


In [ ]:
def make_preprocessor(use_text=True, max_features=120_000):
    parts = []
    if numeric_features:
        num_pipe = Pipeline([
            ("impute", SimpleImputer(strategy="median", add_indicator=True)),
            ("scale", StandardScaler(with_mean=False))
        ])
        parts.append(("num", num_pipe, numeric_features))
    if category_features:
        cat_pipe = Pipeline([
            ("impute", SimpleImputer(strategy="constant", fill_value="__MISSING__")),
            ("onehot", OneHotEncoder(handle_unknown="infrequent_if_exist",
                                     min_frequency=20, max_categories=300))
        ])
        parts.append(("cat", cat_pipe, category_features))
    if use_text:
        parts.append(("title", TfidfVectorizer(
            lowercase=True, strip_accents="unicode", ngram_range=(1, 2),
            min_df=2, max_features=max_features, sublinear_tf=True,
            dtype=np.float32
        ), "title"))
    return ColumnTransformer(parts, remainder="drop", sparse_threshold=1.0)


### 13.2 Benchmark inicial

Regresión logística y SVM lineal combinan texto y tabla; Naive Bayes mide la señal del título por separado; Random Forest y XGBoost usan solo variables estructuradas. XGBoost se omite si no está instalado.


In [ ]:
models = {
    "Logística TF-IDF + tabular": Pipeline([
        ("prep", make_preprocessor(True)),
        ("model", LogisticRegression(C=2.0, solver="liblinear", max_iter=1000))
    ]),
    "SVM lineal TF-IDF + tabular": Pipeline([
        ("prep", make_preprocessor(True)),
        ("model", LinearSVC(C=1.0, max_iter=3000))
    ]),
    "Naive Bayes título": Pipeline([
        ("text", ColumnTransformer([("title", TfidfVectorizer(
            lowercase=True, strip_accents="unicode", ngram_range=(1, 2),
            min_df=2, max_features=120_000, sublinear_tf=True,
            dtype=np.float32
        ), "title")], remainder="drop", sparse_threshold=1.0)),
        ("model", MultinomialNB(alpha=0.15))
    ]),
    "Random Forest tabular": Pipeline([
        ("prep", make_preprocessor(False)),
        ("model", RandomForestClassifier(
            n_estimators=250, min_samples_leaf=3, max_features="sqrt",
            n_jobs=1, random_state=SEED
        ))
    ]),
    "Red neuronal MLP tabular": Pipeline([
        ("prep", make_preprocessor(False)),
        ("model", MLPClassifier(
            hidden_layer_sizes=(64, 32), activation="relu", alpha=0.01,
            batch_size=256, learning_rate_init=0.001, max_iter=100,
            early_stopping=True, n_iter_no_change=8, random_state=SEED
        ))
    ])
}
try:
    from xgboost import XGBClassifier
    models["XGBoost tabular"] = Pipeline([
        ("prep", make_preprocessor(False)),
        ("model", XGBClassifier(
            n_estimators=350, max_depth=6, learning_rate=0.06,
            subsample=0.85, colsample_bytree=0.85, reg_lambda=3.0,
            objective="binary:logistic", eval_metric="logloss",
            tree_method="hist", n_jobs=1, random_state=SEED
        ))
    ])
except ImportError:
    print("XGBoost no está instalado; se omite en esta corrida.")

results, fitted_models, holdout_predictions = [], {}, {}
for name, model in [("Baseline mayoritario", DummyClassifier(strategy="most_frequent")),
                    *models.items()]:
    t0 = perf_counter()
    model.fit(X_dev, y_dev)
    pred = model.predict(X_holdout)
    fitted_models[name] = model
    holdout_predictions[name] = pred
    results.append({
        "modelo": name, "accuracy": accuracy_score(y_holdout, pred),
        "balanced_accuracy": balanced_accuracy_score(y_holdout, pred),
        "macro_f1": f1_score(y_holdout, pred, average="macro"),
        "accuracy_train": accuracy_score(y_dev, model.predict(X_dev)),
        "segundos": perf_counter() - t0
    })
benchmark = pd.DataFrame(results).sort_values("accuracy", ascending=False).reset_index(drop=True)
display(benchmark.round(4))


In [ ]:
best_holdout_name = benchmark.loc[
    benchmark["modelo"] != "Baseline mayoritario", "modelo"
].iloc[0]
best_pred = holdout_predictions[best_holdout_name]
print("Mejor en este holdout:", best_holdout_name)
print(classification_report(y_holdout, best_pred, labels=np.arange(len(classes)),
                            target_names=classes, zero_division=0))
ConfusionMatrixDisplay.from_predictions(
    y_holdout, best_pred, display_labels=classes, normalize="true",
    values_format=".2f", cmap="Blues"
)
plt.title("Holdout estratificado — " + best_holdout_name)
plt.tight_layout()
plt.show()

errors = X_holdout.loc[best_pred != y_holdout].copy()
errors["real"] = label_encoder.inverse_transform(y_holdout[best_pred != y_holdout])
errors["predicho"] = label_encoder.inverse_transform(best_pred[best_pred != y_holdout])
errors["fila_train"] = X_holdout.index[best_pred != y_holdout]
display(errors[[c for c in ["fila_train", "real", "predicho", "title",
                             "category_id", "price_log1p"] if c in errors]].head(30))


In [ ]:
# Comparación pareada: ambos modelos predicen exactamente las mismas filas del holdout.
if {"XGBoost tabular", "Random Forest tabular"}.issubset(holdout_predictions):
    from scipy.stats import binomtest
    xgb_ok = holdout_predictions["XGBoost tabular"] == y_holdout
    rf_ok = holdout_predictions["Random Forest tabular"] == y_holdout
    xgb_only = int(np.sum(xgb_ok & ~rf_ok))
    rf_only = int(np.sum(~xgb_ok & rf_ok))
    discordant = xgb_only + rf_only
    p_value = binomtest(xgb_only, discordant, p=0.5, alternative="two-sided").pvalue if discordant else 1.0
    display(pd.DataFrame([{
        "XGBoost_correcto_RF_incorrecto": xgb_only,
        "RF_correcto_XGBoost_incorrecto": rf_only,
        "diferencia_accuracy": accuracy_score(y_holdout, holdout_predictions["XGBoost tabular"])
                              - accuracy_score(y_holdout, holdout_predictions["Random Forest tabular"]),
        "p_McNemar_exacto": p_value
    }]).round(5))
else:
    print("La prueba pareada requiere que ambos modelos se hayan ejecutado.")


### 13.3 Estabilidad entre particiones y vendedores

La validación cruzada de 3 folds se hace solo en el conjunto de desarrollo. Luego medimos el mejor finalista en una partición donde los vendedores quedan separados. Si el score agrupado baja, hay dependencia de vendedores conocidos; ese resultado sirve como estrés, mientras que el holdout aleatorio sigue siendo la estimación primaria para un test aleatorio.


In [ ]:
grouped_results = []
if "seller_id" in train.columns:
    groups = train.loc[X.index, "seller_id"].astype("string").fillna("__MISSING_SELLER__")
    gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
    gtr, gva = next(gss.split(X, y, groups=groups))
    name = cv_results.iloc[0]["modelo"]
    grouped_model = clone(models[name]).fit(X.iloc[gtr], y[gtr])
    grouped_pred = grouped_model.predict(X.iloc[gva])
    grouped_results.append({
        "modelo": name,
        "accuracy_vendedores_separados": accuracy_score(y[gva], grouped_pred),
        "balanced_accuracy_vendedores_separados": balanced_accuracy_score(y[gva], grouped_pred),
        "n_validacion_vendedores": len(gva)
    })
grouped_validation = pd.DataFrame(grouped_results)
if len(grouped_validation):
    display(grouped_validation.round(4))
else:
    print("No hay seller_id; se omite la partición por vendedores.")


### 13.4 TPE opcional para XGBoost

El TPE se usa solo sobre el conjunto de desarrollo, con los mismos 3 folds y sin consultar el test oficial. El holdout ya se usó para comparar los modelos iniciales, así que cualquier búsqueda adicional debe interpretarse con cautela; el CV y la partición por vendedor son los controles de estabilidad.


In [ ]:
RUN_TPE = False
if RUN_TPE:
    import importlib.util
    if importlib.util.find_spec("optuna") is None:
        raise ImportError("Optuna no está instalado. Ejecutá primero una celda con: %pip install optuna")
    try:
        from xgboost import XGBClassifier
    except ImportError as exc:
        raise ImportError("XGBoost no está instalado en este runtime.") from exc
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)

    def objective(trial):
        params = {
            "n_estimators": trial.suggest_int("n_estimators", 200, 600, step=100),
            "max_depth": trial.suggest_int("max_depth", 3, 8),
            "learning_rate": trial.suggest_float("learning_rate", 0.02, 0.15, log=True),
            "min_child_weight": trial.suggest_float("min_child_weight", 1.0, 10.0, log=True),
            "subsample": trial.suggest_float("subsample", 0.65, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.65, 1.0),
            "reg_lambda": trial.suggest_float("reg_lambda", 0.5, 10.0, log=True)
        }
        candidate = Pipeline([
            ("prep", make_preprocessor(False)),
            ("model", XGBClassifier(
                **params, objective="binary:logistic", eval_metric="logloss",
                tree_method="hist", n_jobs=1, random_state=SEED
            ))
        ])
        return cross_validate(candidate, X_dev, y_dev, cv=cv,
                              scoring="accuracy", n_jobs=1)["test_score"].mean()

    study = optuna.create_study(direction="maximize",
                                sampler=optuna.samplers.TPESampler(seed=SEED))
    study.optimize(objective, n_trials=8, show_progress_bar=True)
    print("Mejor accuracy CV:", round(study.best_value, 4))
    print("Hiperparámetros:", study.best_params)

    tpe_name = "XGBoost tabular (TPE)"
    models[tpe_name] = Pipeline([
        ("prep", make_preprocessor(False)),
        ("model", XGBClassifier(
            **study.best_params, objective="binary:logistic", eval_metric="logloss",
            tree_method="hist", n_jobs=1, random_state=SEED
        ))
    ])
    tpe_model = clone(models[tpe_name]).fit(X_dev, y_dev)
    tpe_pred = tpe_model.predict(X_holdout)
    holdout_predictions[tpe_name] = tpe_pred
    tpe_holdout_accuracy = accuracy_score(y_holdout, tpe_pred)
    print("Accuracy holdout del candidato TPE:", round(tpe_holdout_accuracy, 4))


## 14. Reentrenar y generar la submission

Se toma el primer finalista segun la media de CV y se reentrena con todo train. Si existe `sample_submission.csv`, sus IDs se conservan exactamente como los define Kaggle. Cuando los IDs del test y del template coinciden, las predicciones se alinean por ID; si el test no trae el mismo conjunto, se conserva el orden oficial de filas del test y los IDs del template. Antes de subir, la celda valida encabezados, cantidad de filas, IDs nulos/duplicados y clases.


In [ ]:
selected_model_name = cv_results.iloc[0]["modelo"] if len(cv_results) else best_holdout_name
if RUN_TPE and tpe_holdout_accuracy > accuracy_score(
    y_holdout, holdout_predictions[selected_model_name]
):
    selected_model_name = tpe_name
print("Modelo para el primer envio:", selected_model_name)
final_model = clone(models[selected_model_name]).fit(X, y)
test_pred = label_encoder.inverse_transform(final_model.predict(X_test_model))

if len(test_pred) != len(test):
    raise ValueError(f"Hay {len(test_pred)} predicciones para {len(test)} filas de test.")

# El template de Kaggle es la fuente canonica de los IDs de submission.
sample_path = next(downloaded_dir.rglob("sample_submission.csv"), None)
test_id_col = next((c for c in test.columns if str(c).strip().lower() == "id"), None)

if sample_path is not None:
    sample_columns = pd.read_csv(sample_path, nrows=0).columns.tolist()
    sample_id_col = next((c for c in sample_columns if str(c).strip().lower() == "id"), None)
    if sample_id_col is None:
        raise ValueError(f"El sample_submission no tiene columna ID. Columnas: {sample_columns}")
    # Leer IDs como texto para preservar ceros iniciales y formato oficial.
    sample = pd.read_csv(sample_path, dtype={sample_id_col: "string"})
    if TARGET not in sample.columns or len(sample) != len(test):
        raise ValueError("El sample_submission no coincide con target/filas de test.")

    # No reemplazar los IDs oficiales por los del JSONL.
    official_ids = sample[sample_id_col].copy()
    if official_ids.isna().any() or official_ids.duplicated().any():
        raise ValueError("El sample_submission contiene IDs nulos o duplicados.")

    # Si ambos archivos usan los mismos IDs, alinear explícitamente por ID.
    # Si no, el template y el test se corresponden por orden de filas de la competencia.
    if test_id_col is not None:
        test_ids = test[test_id_col]
        if test_ids.isna().any() or test_ids.duplicated().any():
            raise ValueError("La columna ID de test contiene valores nulos o duplicados.")
        test_key = test_ids.astype(str).reset_index(drop=True)
        official_key = official_ids.astype(str).reset_index(drop=True)
        if set(test_key) == set(official_key):
            pred_by_id = pd.Series(test_pred, index=test_key)
            aligned_pred = official_key.map(pred_by_id).to_numpy()
            print("Predicciones alineadas por ID con sample_submission.csv.")
        else:
            aligned_pred = test_pred
            print("Los IDs del test no coinciden con el template; conservo los IDs oficiales "
                  "y el orden de filas entregado por la competencia.")
    else:
        aligned_pred = test_pred
        print("Test no trae columna ID; uso el orden oficial de filas del template.")

    submission = pd.DataFrame({"ID": official_ids.to_numpy(), TARGET: aligned_pred})
else:
    if test_id_col is None:
        raise KeyError(f"No hay sample_submission ni columna ID en test. Columnas: {test.columns.tolist()}")
    submission = pd.DataFrame({"ID": test[test_id_col].to_numpy(), TARGET: test_pred})
    print("Aviso: no se encontro sample_submission.csv; uso los IDs del test.")

# Contrato de salida que requiere la competencia.
submission = submission[["ID", TARGET]]
if submission.columns.tolist() != ["ID", TARGET]:
    raise ValueError(f"Encabezados inesperados: {submission.columns.tolist()}")
if submission["ID"].isna().any() or submission["ID"].duplicated().any():
    raise ValueError("La submission tiene IDs nulos o duplicados.")
if len(submission) != len(test):
    raise ValueError("La cantidad de filas de la submission no coincide con test.")
if submission[TARGET].isna().any():
    raise ValueError("La submission tiene predicciones vacias.")
valid_labels = set(label_encoder.classes_)
if not set(submission[TARGET].unique()).issubset(valid_labels):
    raise ValueError(f"Hay clases invalidas en condition: {submission[TARGET].unique()}")

output_dir = Path("/content") if Path("/content").exists() else Path.cwd()
submission_path = output_dir / "submission_condition.csv"
submission.to_csv(submission_path, index=False)

# Releer el CSV exportado para comprobar el encabezado y el contenido final.
check = pd.read_csv(submission_path)
if check.columns.tolist() != ["ID", TARGET] or len(check) != len(test):
    raise ValueError("El CSV exportado no cumple el esquema esperado de Kaggle.")
if not check["ID"].astype(str).equals(submission["ID"].astype(str).reset_index(drop=True)):
    raise ValueError("Los IDs cambiaron al exportar el CSV.")
print("Archivo listo:", submission_path, "| filas:", len(check))
print("Columnas:", check.columns.tolist())
display(check.head())
display(check[TARGET].value_counts(normalize=True).rename("proporcion").to_frame())


In [ ]:
try:
    from google.colab import files
    files.download(str(submission_path))
except ImportError:
    print("Submission guardada en:", submission_path)


### Registro de experimentos

Guardar seed, variables incluidas/excluidas, Accuracy, macro-F1, variabilidad de CV, score agrupado, duración y score del leaderboard. Evitar ajustar repetidamente contra el mismo holdout a partir de errores individuales: después de muchas iteraciones también se sobreajusta la validación.


## 15. Auditoría de atributos y posible señal directa del target

Algunas publicaciones pueden contener en attributes un campo como ITEM_CONDITION o “Condición del ítem”. Se audita su cobertura y concordancia con el target antes de entrenar. El campo queda fuera del modelo por defecto porque podría ser una copia directa de condition; la auditoría permite decidir si se trata de una variable permitida por las reglas de la competencia o de fuga.


In [ ]:
import unicodedata

def normalize_audit_text(value):
    value = str(value).strip().lower()
    return "".join(ch for ch in unicodedata.normalize("NFKD", value)
                   if not unicodedata.combining(ch))

def condition_attributes(value):
    obj = parse_nested(value)
    if isinstance(obj, dict):
        obj = [obj]
    if not isinstance(obj, (list, tuple)):
        return []
    found = []
    for item in obj:
        if not isinstance(item, dict):
            continue
        attr_id = normalize_audit_text(item.get("id", ""))
        name = normalize_audit_text(item.get("name", ""))
        group = normalize_audit_text(item.get("attribute_group_name", ""))
        direct_match = (
            "item_condition" in attr_id or "condition" in attr_id
            or name in {"condicion del item", "estado del articulo",
                        "estado del inmueble", "condicion"}
        )
        if direct_match:
            found.append({
                "id": item.get("id"), "name": item.get("name"),
                "value_name": item.get("value_name"), "value_id": item.get("value_id")
            })
    return found

if "attributes" in train:
    audit_rows = []
    for idx, value in train["attributes"].items():
        for item in condition_attributes(value):
            audit_rows.append({"fila": idx, **item})
    condition_attr_audit = pd.DataFrame(audit_rows)
    if len(condition_attr_audit):
        condition_attr_audit["valor_norm"] = condition_attr_audit["value_name"].map(normalize_audit_text)
        condition_attr_audit["valor_mapeado"] = condition_attr_audit["valor_norm"].map({
            "new": "new", "nuevo": "new", "nueva": "new",
            "used": "used", "usado": "used", "usada": "used"
        })
        target_by_idx = train[TARGET].astype(str)
        comparable = condition_attr_audit.drop_duplicates("fila").set_index("fila")
        comparable["target"] = target_by_idx.reindex(comparable.index)
        valid = comparable["valor_mapeado"].notna()
        print("Filas con atributo de condición:", len(comparable), "de", len(train),
              f"({len(comparable)/len(train):.2%})")
        if valid.any():
            print("Concordancia directa con condition:",
                  f"{(comparable.loc[valid, 'valor_mapeado'] == comparable.loc[valid, 'target']).mean():.2%}",
                  "sobre", int(valid.sum()), "filas comparables")
        display(comparable.reset_index().head(30))
    else:
        print("No se encontró un atributo explícito de condición en train.")
else:
    print("No existe la columna attributes.")


## 16. Tabla final de modelos y meta de Accuracy

La meta de 0,94 se trata como objetivo de desarrollo, no como un score garantizado. La tabla consolida los modelos probados y separa el holdout aleatorio, la media/desvío de CV y la prueba por vendedores. El score de Kaggle se puede completar después de enviar la submission; no se usa para elegir hiperparámetros.


In [ ]:
TARGET_ACCURACY = 0.94
KAGGLE_ACCURACY = None  # Completar manualmente tras enviar la submission, por ejemplo 0.88.

summary = benchmark.rename(columns={
    "accuracy": "accuracy_holdout",
    "accuracy_train": "accuracy_entrenamiento"
}).copy()
summary = summary.merge(
    cv_results[["modelo", "accuracy_media", "accuracy_sd"]].rename(columns={
        "accuracy_media": "accuracy_cv_media",
        "accuracy_sd": "accuracy_cv_sd"
    }),
    on="modelo", how="left"
)
if len(grouped_validation):
    summary = summary.merge(
        grouped_validation[["modelo", "accuracy_vendedores_separados"]],
        on="modelo", how="left"
    )
else:
    summary["accuracy_vendedores_separados"] = np.nan

if RUN_TPE and "tpe_name" in globals():
    tpe_row = pd.DataFrame([{
        "modelo": tpe_name,
        "accuracy_holdout": tpe_holdout_accuracy,
        "balanced_accuracy": balanced_accuracy_score(y_holdout, tpe_pred),
        "macro_f1": f1_score(y_holdout, tpe_pred, average="macro"),
        "accuracy_entrenamiento": accuracy_score(y_dev, tpe_model.predict(X_dev)),
        "segundos": np.nan,
        "accuracy_cv_media": study.best_value,
        "accuracy_cv_sd": np.nan,
        "accuracy_vendedores_separados": np.nan
    }])
    summary = pd.concat([summary, tpe_row], ignore_index=True)

summary["distancia_a_0.94_pp"] = (TARGET_ACCURACY - summary["accuracy_holdout"]) * 100
summary["meta_0.94_holdout"] = summary["accuracy_holdout"] >= TARGET_ACCURACY
summary["accuracy_kaggle"] = np.nan
if KAGGLE_ACCURACY is not None and "selected_model_name" in globals():
    summary.loc[summary["modelo"].eq(selected_model_name), "accuracy_kaggle"] = KAGGLE_ACCURACY

summary = summary.sort_values(
    ["accuracy_cv_media", "accuracy_holdout"], ascending=False, na_position="last"
).reset_index(drop=True)
display(summary[[
    "modelo", "accuracy_holdout", "accuracy_cv_media", "accuracy_cv_sd",
    "accuracy_vendedores_separados", "accuracy_entrenamiento",
    "accuracy_kaggle", "distancia_a_0.94_pp", "meta_0.94_holdout"
]].round(4))

best_local_row = summary.loc[summary["modelo"] != "Baseline mayoritario"].iloc[0]
print(f"Mejor modelo segun CV/holdout: {best_local_row['modelo']}")
print(f"Accuracy holdout: {best_local_row['accuracy_holdout']:.4f}")
print(f"Faltan {best_local_row['distancia_a_0.94_pp']:.2f} puntos porcentuales para 0.94.")
if best_local_row["accuracy_holdout"] >= TARGET_ACCURACY:
    print("La meta 0.94 se alcanzo en holdout; confirmar estabilidad antes de enviar.")
else:
    print("La meta 0.94 todavia no se alcanza. Priorizar atributos auditados, nuevas features y validacion; no ajustar repetidamente contra el mismo holdout.")

# Registro acumulado: cada ejecucion queda identificada y no reemplaza corridas previas.
RUN_ID = pd.Timestamp.utcnow().strftime("%Y%m%d_%H%M%S")
FEATURES_VERSION = "tabular_text_features_v1"
LOG_COLUMNS = [
    "run_id", "timestamp_utc", "features_version", "seed", "split",
    "modelo", "accuracy_holdout", "accuracy_cv_media", "accuracy_cv_sd",
    "accuracy_vendedores_separados", "accuracy_entrenamiento",
    "accuracy_kaggle", "distancia_a_0.94_pp"
]
history_rows = summary.copy()
history_rows["run_id"] = RUN_ID
history_rows["timestamp_utc"] = pd.Timestamp.utcnow().isoformat()
history_rows["features_version"] = FEATURES_VERSION
history_rows["seed"] = SEED
history_rows["split"] = "stratified_80_20_seed_42"
history_rows["accuracy_kaggle"] = KAGGLE_ACCURACY
history_rows = history_rows.reindex(columns=LOG_COLUMNS)

log_root = PROJECT_DIR
log_root.mkdir(parents=True, exist_ok=True)
EXPERIMENT_LOG_PATH = log_root / "MercadoLibre_experiment_log.csv"
if EXPERIMENT_LOG_PATH.exists():
    previous_runs = pd.read_csv(EXPERIMENT_LOG_PATH)
    history_rows = pd.concat([previous_runs, history_rows], ignore_index=True)
history_rows.to_csv(EXPERIMENT_LOG_PATH, index=False)
print("Historial acumulado guardado en:", EXPERIMENT_LOG_PATH)

# Tabla final historica: mejor corrida por modelo y version de features, priorizando CV.
historical_best = (
    history_rows.sort_values(
        ["accuracy_cv_media", "accuracy_holdout"], ascending=False, na_position="last"
    )
    .drop_duplicates(["features_version", "modelo"], keep="first")
    .sort_values(["accuracy_cv_media", "accuracy_holdout"], ascending=False, na_position="last")
    .reset_index(drop=True)
)
print("Mejores resultados historicos por modelo y version de variables:")
display(historical_best[[
    "features_version", "modelo", "accuracy_holdout", "accuracy_cv_media",
    "accuracy_cv_sd", "accuracy_vendedores_separados", "accuracy_kaggle",
    "distancia_a_0.94_pp", "run_id"
]].round(4))
